# Quick HH-RLHF-Style Comparison: DPO vs SFT

Use this notebook for a fast qualitative check on 10 prompts from `eval_sets/hh_rlhf_eval_300_seed42.json`. It generates SFT and DPO responses, builds randomized pairwise examples, and displays the outputs side by side.


## Setup


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/ahmad903501/qwen-dpo-alignment.git"
REPO_DIR = Path('/kaggle/working/qwen-dpo-alignment')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}


In [ ]:
# Kaggle already has torch installed. Avoid reinstalling torch.
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps


## Paths


In [ ]:
# If checkpoints are attached from a training notebook, set these to /kaggle/input/... paths.
SFT_MODEL_PATH = "outputs/sft-qwen-0.5b/best"
DPO_MODEL_PATH = "outputs/dpo-qwen-0.5b/best"

# Examples for attached Kaggle training outputs:
# SFT_MODEL_PATH = "/kaggle/input/YOUR_TRAINING_OUTPUT_DATASET/outputs/sft-qwen-0.5b/best"
# DPO_MODEL_PATH = "/kaggle/input/YOUR_TRAINING_OUTPUT_DATASET/outputs/dpo-qwen-0.5b/best"

HH_EVAL_PATH = "eval_sets/hh_rlhf_eval_300_seed42.json"
N_EXAMPLES = 10
OUT_DIR = Path("outputs/hh_quick_compare")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("SFT_MODEL_PATH:", SFT_MODEL_PATH)
print("DPO_MODEL_PATH:", DPO_MODEL_PATH)
print("HH_EVAL_PATH:", HH_EVAL_PATH)


## Generate 10 Responses From Each Model


In [ ]:
!python scripts/generate_alpaca.py \
  --model_name_or_path {SFT_MODEL_PATH} \
  --subset_path {HH_EVAL_PATH} \
  --output_path {OUT_DIR}/sft_hh_eval_10.json \
  --generator_name qwen2.5-0.5b-sft \
  --batch_size 2 \
  --max_new_tokens 256 \
  --limit {N_EXAMPLES} \
  --device cuda:0

!python scripts/generate_alpaca.py \
  --model_name_or_path {DPO_MODEL_PATH} \
  --subset_path {HH_EVAL_PATH} \
  --output_path {OUT_DIR}/dpo_hh_eval_10.json \
  --generator_name qwen2.5-0.5b-dpo \
  --batch_size 2 \
  --max_new_tokens 256 \
  --limit {N_EXAMPLES} \
  --device cuda:0


## Build Pairwise File


In [ ]:
!python scripts/build_pairwise_eval.py \
  --a_outputs {OUT_DIR}/dpo_hh_eval_10.json \
  --b_outputs {OUT_DIR}/sft_hh_eval_10.json \
  --a_name dpo \
  --b_name sft \
  --output_path {OUT_DIR}/dpo_vs_sft_hh_pairs_10.jsonl


## Side-by-Side Review


In [ ]:
import json
from IPython.display import Markdown, display

sft_rows = {row['id']: row for row in json.load(open(OUT_DIR / 'sft_hh_eval_10.json'))}
dpo_rows = {row['id']: row for row in json.load(open(OUT_DIR / 'dpo_hh_eval_10.json'))}
ids = list(dpo_rows.keys())

def esc(text):
    return str(text or '').replace('`', '\`')

for i, item_id in enumerate(ids, start=1):
    dpo = dpo_rows[item_id]
    sft = sft_rows[item_id]
    md = f"""
## {i}. {item_id}

**Category:** `{esc(dpo.get('category'))}`  
**Expected behavior:** {esc(dpo.get('expected_behavior'))}

**Instruction**

{esc(dpo['instruction'])}

### SFT

{esc(sft['output'])}

### DPO

{esc(dpo['output'])}

**Manual winner:** `________`

---
"""
    display(Markdown(md))


## Optional: Judge These 10 With Kaggle Models


In [ ]:
# Uncomment if Kaggle model access is enabled.
# !python scripts/judge_pairwise_kaggle.py \
#   --pairs_path {OUT_DIR}/dpo_vs_sft_hh_pairs_10.jsonl \
#   --output_path {OUT_DIR}/dpo_vs_sft_hh_judged_10.jsonl \
#   --judge_model google/gemini-3.6-flash \
#   --batch_size 5 \
#   --sleep_seconds 5 \
#   --max_retries 3 \
#   --reuse_judgments
